# CASMI26 — dual-encoder reranker, offline inference

This notebook writes `submission.csv` for the *Enveda CASMI 2026* competition, using the model trained in
`mo_graph/dual_encoder_reranker.ipynb` (ECSE 552 project, branch `mo_graph`).

For each test molecule:

1. **Candidate pool.** Take every catalog structure within 10 ppm of the neutral mass implied by its spectra. The
   catalog holds the 275,810 structures of the training set. Pad the pool to 25 with the nearest masses.
2. **Score each spectrum against each candidate.**
   * The spectrum encoder is MIST-style: peaks are annotated with subformulae of the candidate's formula and encoded
     by a transformer.
   * The candidate's atom/bond graph goes through a GINE graph neural network.
   * A matching head combines both embeddings with mass error and a fingerprint likelihood.
3. **Combine the spectra.** Average per-spectrum log-softmax scores. The 25 best candidates, best first, form the
   submission.

**Inputs** (attach both; internet off):

* the competition data;
* the private dataset `casmi26-dual-reranker`, exported by the training notebook. It contains:
  * `dualrerank.py` (the model/scoring code);
  * `catalog_arrays.npz` and `catalog_meta.parquet`;
  * `dual_mist_gnn.pt` and `config.json`.

Only numpy, pandas, pyarrow and torch are needed (all preinstalled on Kaggle); RDKit is not.

**Limit.** This is retrieval, so it can only find structures that are in the catalog (Kaggle classes 1–2). Novel
structures (class 3) need de novo generation.

In [ ]:
import json, sys, time
from pathlib import Path

import numpy as np
import pandas as pd
import torch

ON_KAGGLE = Path("/kaggle/input").exists()
# Kaggle: everything is mounted under /kaggle/input. Locally: run from mo_graph/kaggle/ after exporting the bundle.
ROOTS = [Path("/kaggle/input")] if ON_KAGGLE else [Path.cwd() / "bundle",
                                                   Path.cwd().parents[1] / "data" / "enveda-CASMI26-molecule-id-mass-spectra"]
OUT_DIR = Path("/kaggle/working") if ON_KAGGLE else Path.cwd()


def find(name):
    for root in ROOTS:
        hits = sorted(root.rglob(name)) if root.exists() else []
        if hits:
            return hits[0]
    raise FileNotFoundError(f"{name} not found under {ROOTS} (is the dataset attached?)")


BUNDLE = find("dualrerank.py").parent
sys.path.insert(0, str(BUNDLE))
import dualrerank as dr

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
cfg = json.loads((BUNDLE / "config.json").read_text())
print(f"bundle: {BUNDLE}\ndevice: {DEVICE}  torch {torch.__version__}")

In [ ]:
t0 = time.time()
cat = dr.Catalog(dr.load_catalog(BUNDLE), DEVICE)
test = pd.read_parquet(find("test.parquet")).reset_index(drop=True)
sp = dr.Spectra(test, cfg["adduct_vocab"], cfg["max_peaks"])
model = dr.DualReranker(True, n_adducts=len(cfg["adduct_vocab"]), **cfg["model"]).to(DEVICE)
model.load_state_dict(torch.load(BUNDLE / "dual_mist_gnn.pt", map_location=DEVICE, weights_only=True))
model.eval(); model.name = "dual"
scorer = dr.Scorer(cat, sp, cfg["ppm"], cfg["annot_ppm"], cfg["annot_abs"])
zg = scorer.embed_catalog(model)           # graph embeddings of all catalog structures, computed once
print(f"{cat.N:,} catalog structures, {len(test):,} test spectra of {test.molecule_id.nunique():,} molecules "
      f"({time.time() - t0:.0f}s)")
assert np.isfinite(sp.neutral).all(), "unexpected adduct in test.parquet"

In [ ]:
t0 = time.time()
rows = []
for mol, mq in pd.Series(np.arange(len(test))).groupby(test.molecule_id.values).apply(list).items():
    pool = scorer.molecule_pool(mq, fill=25)
    score = dr.aggregate([d[model.name] for d in scorer.score_pools(model, [(q, pool) for q in mq], zg)])
    top = pool[np.argsort(-score, kind="stable")[:25]]
    rows.append((mol, ";".join(cat.smiles[top])))
sub = pd.DataFrame(rows, columns=["molecule_id", "smiles"])

try:                                       # same molecules and order as sample_submission.csv
    ref = pd.read_csv(find("sample_submission.csv"))
    assert set(ref.molecule_id) == set(sub.molecule_id)
    sub = sub.set_index("molecule_id").loc[ref.molecule_id].reset_index()
except FileNotFoundError:
    pass
assert sub.molecule_id.is_unique and sub.smiles.notna().all()
assert sub.smiles.str.split(";").map(len).max() <= 25
sub.to_csv(OUT_DIR / "submission.csv", index=False)
print(f"submission.csv: {len(sub)} molecules, {sub.smiles.str.split(';').map(len).mean():.1f} guesses each "
      f"({time.time() - t0:.0f}s)")
sub.head()